# PySpark practice 11: Spark SQL and the catalog

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-11-spark-sql-and-catalog.ipynb)

14 exercises on parameterized SQL, views, the catalog, managed and external tables, DDL, subqueries, PIVOT and dynamic SQL. Each exercise names the [reference entries](pyspark-reference-11-spark-sql-and-catalog.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 11](pyspark-reference-11-spark-sql-and-catalog.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. The shared tables are registered as temporary views.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import os

orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")

## Exercises

5 easy, 9 medium, 0 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Query a temp view with named parameters](#exercise-1-query-a-temp-view-with-named-parameters) | Easy | [`spark.sql(sqlQuery, args=None, **kwargs)`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sparksqlsqlquery-argsnone-kwargs) |
| 2 | [Pass DataFrames into SQL without views](#exercise-2-pass-dataframes-into-sql-without-views) | Easy | [`spark.sql(sqlQuery, args=None, **kwargs)`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sparksqlsqlquery-argsnone-kwargs) |
| 3 | [Share a view across sessions](#exercise-3-share-a-view-across-sessions) | Medium | [Temporary and global views](pyspark-reference-11-spark-sql-and-catalog.ipynb#temporary-and-global-views) |
| 4 | [Inventory a database with the catalog API](#exercise-4-inventory-a-database-with-the-catalog-api) | Medium | [`spark.catalog`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sparkcatalog) |
| 5 | [What DROP TABLE deletes](#exercise-5-what-drop-table-deletes) | Medium | [Managed and external tables](pyspark-reference-11-spark-sql-and-catalog.ipynb#managed-and-external-tables) |
| 6 | [Partitioned table with a partition overwrite](#exercise-6-partitioned-table-with-a-partition-overwrite) | Medium | [Common DDL and DML](pyspark-reference-11-spark-sql-and-catalog.ipynb#common-ddl-and-dml) |
| 7 | [Customers with a return, in SQL](#exercise-7-customers-with-a-return-in-sql) | Easy | [CTEs, subqueries and lateral column aliases](pyspark-reference-11-spark-sql-and-catalog.ipynb#ctes-subqueries-and-lateral-column-aliases) |
| 8 | [Share of revenue with a CTE and scalar subquery](#exercise-8-share-of-revenue-with-a-cte-and-scalar-subquery) | Medium | [CTEs, subqueries and lateral column aliases](pyspark-reference-11-spark-sql-and-catalog.ipynb#ctes-subqueries-and-lateral-column-aliases) |
| 9 | [Pivot statuses in SQL](#exercise-9-pivot-statuses-in-sql) | Easy | [Generators, higher-order functions and PIVOT in SQL](pyspark-reference-11-spark-sql-and-catalog.ipynb#generators-higher-order-functions-and-pivot-in-sql) |
| 10 | [Explode and transform arrays in SQL](#exercise-10-explode-and-transform-arrays-in-sql) | Medium | [Generators, higher-order functions and PIVOT in SQL](pyspark-reference-11-spark-sql-and-catalog.ipynb#generators-higher-order-functions-and-pivot-in-sql) |
| 11 | [Dynamic table names with IDENTIFIER](#exercise-11-dynamic-table-names-with-identifier) | Medium | [Dynamic SQL: `IDENTIFIER()`, variables and `EXECUTE IMMEDIATE`](pyspark-reference-11-spark-sql-and-catalog.ipynb#dynamic-sql-identifier-variables-and-execute-immediate) |
| 12 | [Parameterize a SQL job with variables](#exercise-12-parameterize-a-sql-job-with-variables) | Medium | [Dynamic SQL: `IDENTIFIER()`, variables and `EXECUTE IMMEDIATE`](pyspark-reference-11-spark-sql-and-catalog.ipynb#dynamic-sql-identifier-variables-and-execute-immediate) |
| 13 | [A reusable SQL function](#exercise-13-a-reusable-sql-function) | Medium | [SQL functions: `CREATE FUNCTION`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sql-functions-create-function) |
| 14 | [Unpivot a wide table in SQL](#exercise-14-unpivot-a-wide-table-in-sql) | Easy | [Generators, higher-order functions and PIVOT in SQL](pyspark-reference-11-spark-sql-and-catalog.ipynb#generators-higher-order-functions-and-pivot-in-sql) |

## Exercise 1: Query a temp view with named parameters

**Difficulty:** Easy · about 5 min · **Practises:** [`spark.sql(sqlQuery, args=None, **kwargs)`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sparksqlsqlquery-argsnone-kwargs)

*Parameter markers keep user input out of the SQL text.*

Using the `orders` view and named parameters `:status` = `shipped` and `:min` = 90, return `order_id` and `amount` of matching orders with amount at least `:min`, ordered by `order_id`.

In [ ]:
spark.sql("SELECT order_id, status, amount FROM orders").show()

**Expected output**

| order_id | amount |
| --- | --- |
| 6 | 149.00 |
| 9 | 1180.00 |
| 12 | 96.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("SELECT order_id, amount FROM orders WHERE status = :status AND amount >= :min ORDER BY order_id", args={"status": "shipped", "min": 90})
result.show()

## Exercise 2: Pass DataFrames into SQL without views

**Difficulty:** Easy · about 5 min · **Practises:** [`spark.sql(sqlQuery, args=None, **kwargs)`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sparksqlsqlquery-argsnone-kwargs)

*Mixing Python and SQL cleanly.*

Without registering a view, join the DataFrame `recent` (orders from 2026-01-12 onwards) to `customers` in SQL using `{name}` arguments. Return `order_id`, `name`, ordered by `order_id`.

In [ ]:
recent = orders.where("order_ts >= '2026-01-12'")
recent.select("order_id", "customer_id").show()

**Expected output**

| order_id | name |
| --- | --- |
| 9 | Delta Grocers |
| 10 | Echo Supplies |
| 11 | Foxtrot Foods |
| 12 | Foxtrot Foods |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("SELECT r.order_id, c.name FROM {r} r JOIN {c} c ON r.customer_id = c.customer_id ORDER BY r.order_id", r=recent, c=customers)
result.show()

## Exercise 3: Share a view across sessions

**Difficulty:** Medium · about 10 min · **Practises:** [Temporary and global views](pyspark-reference-11-spark-sql-and-catalog.ipynb#temporary-and-global-views)

*Notebooks sharing a cluster need global temp views; plain temp views are per session.*

Register `customers` as global temp view `cust_g` and as temp view `cust_t`. From a new session, return `view` and `visible` for `global_temp.cust_g` and `cust_t`.

In [ ]:
other = spark.newSession()

**Expected output**

| view | visible |
| --- | --- |
| global_temp.cust_g | true |
| cust_t | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
customers.createOrReplaceGlobalTempView("cust_g")
customers.createOrReplaceTempView("cust_t")
result = spark.createDataFrame([("global_temp.cust_g", other.catalog.tableExists("global_temp.cust_g")), ("cust_t", other.catalog.tableExists("cust_t"))], "view string, visible boolean")
result.show()

## Exercise 4: Inventory a database with the catalog API

**Difficulty:** Medium · about 10 min · **Practises:** [`spark.catalog`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sparkcatalog)

*Data platform automation lists tables and checks schemas before running jobs.*

Create database `p11` with a managed table `orders_t` (from `orders`) and a view `paid_v` (SQL `CREATE VIEW` over the table). Return `name` and `table_type` for every non-temporary object in `p11` (`listTables` also returns the session's temporary views), ordered by `name`.

In [ ]:
spark.sql("DROP DATABASE IF EXISTS p11 CASCADE")

**Expected output**

| name | table_type |
| --- | --- |
| orders_t | MANAGED |
| paid_v | VIEW |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.sql("CREATE DATABASE p11")
orders.write.saveAsTable("p11.orders_t")
spark.sql("CREATE VIEW p11.paid_v AS SELECT * FROM p11.orders_t WHERE status = 'paid'")
result = spark.createDataFrame([(t.name, t.tableType) for t in spark.catalog.listTables("p11") if not t.isTemporary], "name string, table_type string").orderBy("name")
result.show()

## Exercise 5: What DROP TABLE deletes

**Difficulty:** Medium · about 10 min · **Practises:** [Managed and external tables](pyspark-reference-11-spark-sql-and-catalog.ipynb#managed-and-external-tables)

*Dropping a managed table deletes production data; an external one does not.*

Create an external table over `ext_path` and a managed table from `orders`, drop both, and return `table` and `files_remain`.

In [ ]:
ext_path = f"{WORK_DIR}/p11_ext"
orders.write.mode("overwrite").parquet(ext_path)
for t in ["p11_ext", "p11_managed"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

**Expected output**

| table | files_remain |
| --- | --- |
| external | true |
| managed | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.sql(f"CREATE TABLE p11_ext USING parquet LOCATION '{ext_path}'")
spark.sql("CREATE TABLE p11_managed USING parquet AS SELECT * FROM orders")
managed_path = spark.sql("DESCRIBE TABLE EXTENDED p11_managed").where("col_name = 'Location'").first().data_type.replace("file:", "")
spark.sql("DROP TABLE p11_ext")
spark.sql("DROP TABLE p11_managed")
result = spark.createDataFrame([("external", os.path.exists(ext_path)), ("managed", os.path.exists(managed_path))], "table string, files_remain boolean")
result.show()

## Exercise 6: Partitioned table with a partition overwrite

**Difficulty:** Medium · about 15 min · **Practises:** [Common DDL and DML](pyspark-reference-11-spark-sql-and-catalog.ipynb#common-ddl-and-dml)

*SQL-first pipelines load tables with INSERT OVERWRITE per partition.*

Create table `p11_daily (order_id INT, amount DECIMAL(12,2), day DATE)` partitioned by `day`, load all orders, then overwrite partition 2026-01-12 with only order 9. Return `day` and `orders`, ordered by `day`.

In [ ]:
spark.sql("DROP TABLE IF EXISTS p11_daily")

**Expected output**

| day | orders |
| --- | --- |
| 2026-01-05 | 3 |
| 2026-01-06 | 1 |
| 2026-01-07 | 2 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 1 |
| 2026-01-13 | 1 |
| 2026-01-14 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.sql("CREATE TABLE p11_daily (order_id INT, amount DECIMAL(12,2), day DATE) USING parquet PARTITIONED BY (day)")
spark.sql("INSERT INTO p11_daily SELECT order_id, amount, to_date(order_ts) FROM orders")
spark.sql("INSERT OVERWRITE p11_daily PARTITION (day = '2026-01-12') SELECT order_id, amount FROM orders WHERE order_id = 9")
result = spark.sql("SELECT day, count(*) AS orders FROM p11_daily GROUP BY day ORDER BY day")
result.show()

## Exercise 7: Customers with a return, in SQL

**Difficulty:** Easy · about 5 min · **Practises:** [CTEs, subqueries and lateral column aliases](pyspark-reference-11-spark-sql-and-catalog.ipynb#ctes-subqueries-and-lateral-column-aliases)

*EXISTS subqueries are the SQL form of a semi join.*

In SQL, return `customer_id` and `name` of customers with at least one `returned` order using `EXISTS`, ordered by `customer_id`.

In [ ]:
print("views: orders, customers")

**Expected output**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 4 | Delta Grocers |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("""
    SELECT customer_id, name FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'returned')
    ORDER BY customer_id
""")
result.show()

## Exercise 8: Share of revenue with a CTE and scalar subquery

**Difficulty:** Medium · about 10 min · **Practises:** [CTEs, subqueries and lateral column aliases](pyspark-reference-11-spark-sql-and-catalog.ipynb#ctes-subqueries-and-lateral-column-aliases)

*Analysts' SQL is full of CTEs; read and write them fluently.*

In SQL with a CTE, return `customer_id`, `revenue` and `pct` (share of total revenue, rounded to 1, using a lateral column alias for revenue), ordered by `revenue` descending.

In [ ]:
print("view: orders")

**Expected output**

| customer_id | revenue | pct |
| --- | --- | --- |
| 4 | 1269.90 | 47.8 |
| 3 | 459.00 | 17.3 |
| 2 | 315.25 | 11.9 |
| 6 | 306.40 | 11.5 |
| 1 | 234.50 | 8.8 |
| 5 | 74.00 | 2.8 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("""
    WITH per_customer AS (SELECT customer_id, sum(amount) AS revenue FROM orders GROUP BY customer_id)
    SELECT customer_id, revenue, round(revenue * 100 / (SELECT sum(revenue) FROM per_customer), 1) AS pct
    FROM per_customer ORDER BY revenue DESC
""")
result.show()

## Exercise 9: Pivot statuses in SQL

**Difficulty:** Easy · about 5 min · **Practises:** [Generators, higher-order functions and PIVOT in SQL](pyspark-reference-11-spark-sql-and-catalog.ipynb#generators-higher-order-functions-and-pivot-in-sql)

*SQL PIVOT for wide reports.*

In SQL, return one row per customer with order counts per status as columns `paid`, `placed`, `shipped`, `returned` (null when none), ordered by `customer_id`.

In [ ]:
print("view: orders")

**Expected output**

| customer_id | paid | placed | shipped | returned |
| --- | --- | --- | --- | --- |
| 1 | 1 | null | 1 | 1 |
| 2 | 1 | 1 | null | null |
| 3 | 1 | null | 1 | null |
| 4 | null | null | 1 | 1 |
| 5 | 1 | null | null | null |
| 6 | null | 1 | 1 | null |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("""
    SELECT * FROM (SELECT customer_id, status, order_id FROM orders)
    PIVOT (count(order_id) FOR status IN ('paid' AS paid, 'placed' AS placed, 'shipped' AS shipped, 'returned' AS returned))
    ORDER BY customer_id
""")
result.show()

## Exercise 10: Explode and transform arrays in SQL

**Difficulty:** Medium · about 10 min · **Practises:** [Generators, higher-order functions and PIVOT in SQL](pyspark-reference-11-spark-sql-and-catalog.ipynb#generators-higher-order-functions-and-pivot-in-sql)

*Lambda syntax in SQL mirrors the Python higher-order functions.*

From `carts` (registered as a view), return `cart_id`, `item` (one row per item, via `LATERAL VIEW explode`) and `total_qty` (sum of the cart's quantities via `aggregate`), ordered by `cart_id`, `item`.

In [ ]:
spark.createDataFrame([(1, ["tea", "milk"], [2, 1]), (2, ["bread"], [3])], "cart_id int, items array<string>, qty array<int>").createOrReplaceTempView("carts")
spark.table("carts").show()

**Expected output**

| cart_id | item | total_qty |
| --- | --- | --- |
| 1 | milk | 3 |
| 1 | tea | 3 |
| 2 | bread | 3 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("""
    SELECT cart_id, item, aggregate(qty, 0, (acc, q) -> acc + q) AS total_qty
    FROM carts LATERAL VIEW explode(items) e AS item
    ORDER BY cart_id, item
""")
result.show()

## Exercise 11: Dynamic table names with IDENTIFIER

**Difficulty:** Medium · about 10 min · **Practises:** [Dynamic SQL: `IDENTIFIER()`, variables and `EXECUTE IMMEDIATE`](pyspark-reference-11-spark-sql-and-catalog.ipynb#dynamic-sql-identifier-variables-and-execute-immediate)

*Generic jobs run the same SQL over many tables.*

For each view in `tables`, count rows with `IDENTIFIER(:tbl)`. Return `table` and `row_count`, in list order.

In [ ]:
tables = ["orders", "customers", "order_items", "products"]

**Expected output**

| table | row_count |
| --- | --- |
| orders | 12 |
| customers | 8 |
| order_items | 17 |
| products | 13 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rows = [(t, spark.sql("SELECT count(*) AS n FROM IDENTIFIER(:tbl)", args={"tbl": t}).first().n) for t in tables]
result = spark.createDataFrame(rows, "table string, row_count bigint")
result.show()

## Exercise 12: Parameterize a SQL job with variables

**Difficulty:** Medium · about 10 min · **Practises:** [Dynamic SQL: `IDENTIFIER()`, variables and `EXECUTE IMMEDIATE`](pyspark-reference-11-spark-sql-and-catalog.ipynb#dynamic-sql-identifier-variables-and-execute-immediate)

*SQL-only pipelines pass run dates as variables.*

Declare variable `run_date` = 2026-01-12, then return `status` and `orders` for orders on that UTC date, using the variable in the query, ordered by `status`.

In [ ]:
print("view: orders")

**Expected output**

| status | orders |
| --- | --- |
| paid | 1 |
| shipped | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.sql("DECLARE OR REPLACE VARIABLE run_date DATE DEFAULT DATE'2026-01-12'")
result = spark.sql("SELECT status, count(*) AS orders FROM orders WHERE to_date(order_ts) = run_date GROUP BY status ORDER BY status")
result.show()

## Exercise 13: A reusable SQL function

**Difficulty:** Medium · about 10 min · **Practises:** [SQL functions: `CREATE FUNCTION`](pyspark-reference-11-spark-sql-and-catalog.ipynb#sql-functions-create-function)

*SQL UDFs encapsulate business rules without the cost of Python UDFs.*

Create temporary SQL function `tier(amount DECIMAL(12,2))` returning `large` (>= 200), `medium` (>= 100) or `small`. Return `tier` and `orders` (count of orders per tier), ordered by `tier`.

In [ ]:
print("view: orders")

**Expected output**

| tier | orders |
| --- | --- |
| large | 4 |
| medium | 2 |
| small | 6 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TEMPORARY FUNCTION tier(amount DECIMAL(12,2)) RETURNS STRING
    RETURN CASE WHEN amount >= 200 THEN 'large' WHEN amount >= 100 THEN 'medium' ELSE 'small' END
""")
result = spark.sql("SELECT tier(amount) AS tier, count(*) AS orders FROM orders GROUP BY 1 ORDER BY 1")
result.show()

## Exercise 14: Unpivot a wide table in SQL

**Difficulty:** Easy · about 5 min · **Practises:** [Generators, higher-order functions and PIVOT in SQL](pyspark-reference-11-spark-sql-and-catalog.ipynb#generators-higher-order-functions-and-pivot-in-sql)

*Spreadsheet-shaped inputs become long tables.*

Unpivot `targets_wide` (view) into `region`, `quarter`, `target`, ordered by `region`, `quarter`.

In [ ]:
spark.createDataFrame([("North", 100, 120), ("South", 80, 90)], "region string, q1 int, q2 int").createOrReplaceTempView("targets_wide")
spark.table("targets_wide").show()

**Expected output**

| region | quarter | target |
| --- | --- | --- |
| North | q1 | 100 |
| North | q2 | 120 |
| South | q1 | 80 |
| South | q2 | 90 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("SELECT region, quarter, target FROM targets_wide UNPIVOT (target FOR quarter IN (q1, q2)) ORDER BY region, quarter")
result.show()

**Next:** look anything up in [PySpark reference 11: Spark SQL and the catalog](pyspark-reference-11-spark-sql-and-catalog.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).